In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

df = pd.read_csv('nacimientos_procesado.csv')
print(f"Dataset cargado: {df.shape}")
print(f"Columnas: {list(df.columns)}")

Dataset cargado: (1413203, 21)
Columnas: ['EDAD', 'ESCOLARIDAD', 'TRABAJAACTUALMENTE', 'ESTADOCONYUGAL', 'NUMEROEMBARAZOS', 'ATENCIONPRENATAL', 'TRIMESTREPRIMERCONSULTA', 'TOTALCONSULTAS', 'AFILIACION', 'LUGARNACIMIENTO', 'PESO', 'TALLA', 'EDADGESTACIONAL', 'SEXO', 'APGAR', 'escolaridad_desc', 'categoria_peso', 'madre_adolescente', 'atencion_adecuada', 'sin_derechohabiencia', 'prematuro']


In [29]:
variables_clustering = [
    'EDAD',                    
    'ESCOLARIDAD',            
    'NUMEROEMBARAZOS',        
    'TOTALCONSULTAS',        
    'PESO',                   
    'TALLA',                  
    'EDADGESTACIONAL',       
    'APGAR',                  
    'AFILIACION',             
    'LUGARNACIMIENTO',       
    'TRABAJAACTUALMENTE',     
    'ESTADOCONYUGAL'          
]

df_select = df[variables_clustering].copy()

print(f"Registros: {len(df_select)}")
print(f"\nPrimeras filas:")
print(df_select.head())

Registros: 1413203

Primeras filas:
   EDAD  ESCOLARIDAD  NUMEROEMBARAZOS  TOTALCONSULTAS  PESO  TALLA  \
0    19            1                1             8.0  2820     50   
1    29           31                1             6.0  9999     45   
2    22           51                2            12.0  3020     45   
3    29           31                2             8.0  2940     49   
4    29           51                5            99.0  2660     47   

   EDADGESTACIONAL  APGAR  AFILIACION  LUGARNACIMIENTO  TRABAJAACTUALMENTE  \
0               40     10           1               14                   2   
1               41      9           1               14                   2   
2               40     10           1               14                   2   
3               39     10           1               14                   2   
4               35      9           1               14                   2   

   ESTADOCONYUGAL  
0               4  
1               5  
2             

In [30]:
# Revisar valores faltantes
print(f"\nTotal de registros: {len(df_select)}")
print(f"Registros sin faltantes: {df_select.dropna().shape[0]}")

porcentaje_completos = (df_select.dropna().shape[0] / len(df_select)) * 100
print(f"\nPorcentaje de registros completos: {porcentaje_completos:.2f}%")

if porcentaje_completos >= 95:
    print("\n Eliminando filas con faltantes")
    df_cluster = df_select.dropna()
else:
    print("\n Imputando valores faltantes")
    for col in df_select.columns:
        if df_select[col].isnull().sum() > 0:
            mediana = df_select[col].median()
            df_select[col].fillna(mediana, inplace=True)
    df_cluster = df_select




Total de registros: 1413203
Registros sin faltantes: 1412415

Porcentaje de registros completos: 99.94%

 Eliminando filas con faltantes


In [31]:
# Identificar variables tipo object (texto/categóricas)

columnas_object = df_cluster.select_dtypes(include='object').columns

if len(columnas_object) > 0:
    print(f"\n Codificando {len(columnas_object)} columnas categóricas...")
    le = LabelEncoder()
    for col in columnas_object:
        df_cluster[col] = le.fit_transform(df_cluster[col].astype(str))
        print(f" {col} codificado")
else:
    print("\n No hay columnas categóricas - todas ya son numéricas")




 No hay columnas categóricas - todas ya son numéricas


In [32]:
# Escalar todas las variables a la misma escala (0 a 1)
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_cluster)

df_scaled = pd.DataFrame(df_scaled, columns=df_cluster.columns)

print("\nESTADÍSTICAS ANTES DEL ESCALADO:")
print(df_cluster.describe().round(2))
print("\nESTADÍSTICAS DESPUÉS DEL ESCALADO:")
print(df_scaled.describe().round(2))


ESTADÍSTICAS ANTES DEL ESCALADO:
             EDAD  ESCOLARIDAD  NUMEROEMBARAZOS  TOTALCONSULTAS        PESO  \
count  1412415.00   1412415.00       1412415.00      1412415.00  1412415.00   
mean        26.38        62.54             2.17            7.96     3429.57   
std          6.43        19.43             1.26            5.21     1562.48   
min         10.00         0.00             1.00            0.00      350.00   
25%         21.00        51.00             1.00            6.00     2850.00   
50%         26.00        71.00             2.00            8.00     3130.00   
75%         31.00        72.00             3.00           10.00     3450.00   
max         60.00       132.00            99.00           99.00     9999.00   

            TALLA  EDADGESTACIONAL       APGAR  AFILIACION  LUGARNACIMIENTO  \
count  1412415.00       1412415.00  1412415.00  1412415.00       1412415.00   
mean        49.44            38.49        9.33        5.64             6.10   
std          2.92

Calculando métricas para diferentes números de clusters...
